# CartPole PPO from Scratch

## 1. Introduction

This notebook documents a **from-scratch implementation of Proximal Policy Optimization (PPO)**,
applied to the classic `CartPole-v1` environment from [Gymnasium](https://gymnasium.farama.org/).

No deep learning frameworks were used — only **NumPy**. Every component (neural network, backpropagation,
optimizer, rollout buffer) was written by hand with the explicit goal of understanding the impact
of each design choice, rather than treating them as black-box primitives.

### Why from scratch?

Libraries like PyTorch or Stable-Baselines3 make it trivially easy to train a PPO agent in a
dozen lines of code. But that ease comes at a cost: the internals — how gradients flow through
the clipped objective, why the advantage needs to be normalised, what Adam is actually doing to
each gradient — remain opaque.

Re-implementing everything from first principles forces every design decision to be made
consciously and understood deeply. This notebook is the written record of that process.

### What is PPO?

PPO ([Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) is a policy-gradient algorithm
that improves training stability by constraining how much the policy can change in a single update.
It does so through a **clipped surrogate objective** that prevents excessively large gradient steps,
without requiring the complex second-order machinery of TRPO.

The variant implemented here uses:

| Component | Choice |
|---|---|
| Policy | Separate Actor network (softmax output) |
| Value function | Separate Critic network (scalar output) |
| Advantage estimation | GAE-λ (Generalized Advantage Estimation) |
| Optimizer | Adam (implemented from scratch) |
| Environment | `CartPole-v1` — 4D continuous state, 2 discrete actions |

### Notebook structure

| Section | Content |
|---|---|
| **2. Architecture** | Network design, weight initialisation, Actor vs Critic |
| **3. Mathematics** | Derivations: softmax, GAE-λ, PPO gradient, backprop, Adam |
| **4. Implementation** | Annotated code for the key components |
| **5. Experiments** | Hyperparameter sweeps and result analysis |
| **6. Conclusions** | Findings, limitations, next steps |

## 2. Architecture

### 2.1 The environment

`CartPole-v1` presents the agent with a **4-dimensional state vector** at each timestep:

| Index | Variable | Range |
|---|---|---|
| 0 | Cart position | $[-4.8,\ 4.8]$ |
| 1 | Cart velocity | $(-\infty, +\infty)$ |
| 2 | Pole angle | $[-0.418,\ 0.418]$ rad |
| 3 | Pole angular velocity | $(-\infty, +\infty)$ |

The agent must choose between **2 discrete actions** (push left / push right) to keep the pole balanced.
An episode ends when the pole falls beyond ±12° or the cart moves beyond ±2.4 units.
The maximum achievable reward per episode is **500**.

### 2.2 Separate Actor and Critic networks

The implementation uses **two independent networks** — one for the policy (Actor) and one for the
value function (Critic). An alternative design is the **shared backbone** architecture, where both
heads branch off a common set of layers that process the input jointly.

The separate-network design was chosen here for a practical reason: it keeps the two gradient
flows completely independent, which makes the implementation easier to reason about and debug.
In a shared backbone, the policy gradient and the value gradient both flow through the same weights
simultaneously, which can create interference and requires careful tuning of the loss coefficients.

<div style="text-align: center;">
  <img src="images/actor_critic_architecture.svg" alt="Actor-Critic Architecture" width="600"/>
</div>


### 2.3 Activation function: tanh

Hidden layers use **tanh** as the activation function:

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

This choice was made to ensure training stability, since `tanh` is **bounded** in $(-1, +1)$, whereas 
ReLU has no upper limitations. 
In a reinforcement learning setting, where the behavior of the agent is unsteady, an unbounded activation
could lead to a highly unstable learning.

### 2.4 Output layers

**Critic** — single linear output unit, no activation. Since the value function $V(s) \in \mathbb{R}$
could assume any values, an activation function could artificially constrain it.

**Actor** — also a linear output, producing raw **logits** $z \in \mathbb{R}^2$, one per action.
The softmax is applied separately, *after* the forward pass:

$$\pi(a \mid s) = \text{softmax}(z)_a = \frac{e^{z_a}}{\sum_{a'} e^{z_{a'}}}$$

### 2.5 Weight initialisation

Weights are initialised with a uniform distribution scaled by the fan-in:

$$W \sim \mathcal{U}\left(-\frac{1}{\sqrt{n_{neuron}}},\ \frac{1}{\sqrt{n_{neuron}}}\right)$$

This is a simplified variant of **LeCun initialisation**. Where $n_{neuron}$ is the number of neurons of the layer  

## 3. Mathematics

This section derives from first principles every formula that appears in the implementation.
The goal is not just to state the results, but to show where they come from — so that every
line of backpropagation code has a clear mathematical justification.

---

### 3.1 Softmax and log-probabilities

#### Definition

Given the actor's raw output (logits) $z \in \mathbb{R}^{|\mathcal{A}|}$, the softmax converts them
into a valid probability distribution over actions:

$$\pi(a \mid s) = \text{softmax}(z)_a = \frac{e^{z_a}}{\sum_{a'} e^{z_{a'}}}$$

To overcome possible overflows, the standard fix is to subtract the
maximum logit before exponentiating — this does not change the result because the constant cancels out:

$$\text{softmax}(z)_a
= \frac{e^{z_a - c}}{\sum_{a'} e^{z_{a'} - c}}, \quad c = \max_a z_a$$


After the shift, all exponents are $\leq 0$, so the largest value in the numerator is $e^0 = 1$.
No overflow is possible.

#### Log-probability

PPO works with log-probabilities rather than raw probabilities. This is for two reasons:
numerical stability (probabilities can be very small, logs keep them in a manageable range)
and convenience (the probability ratio $r_t = \pi_{\theta}(a|s) / \pi_{\theta_{\text{old}}}(a|s)$
becomes a simple subtraction in log space).

Taking the log of the softmax:

$$\log \pi(a \mid s) = z_a - \log \sum_{a'} e^{z_{a'}}$$

In the implementation, the log-probability of the chosen action $a$ is stored at each timestep:

```python
log_prob = np.log(action_probs[0, action])
```

#### Probability ratio

At update time, the ratio between the new policy $\pi_\theta$ and the old policy $\pi_{\theta_{\text{old}}}$
is computed as:

$$r_t = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}
= e^{\log \pi_\theta(a_t \mid s_t) - \log \pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

```python
ratios = np.exp(new_log_probs - old_log_probs)
```

When $r_t = 1$ the policy has not changed. When $r_t > 1$ the new policy assigns higher
probability to the action than the old one did, and vice versa.

### 3.2 Generalized Advantage Estimation (GAE-λ)

Policy gradient methods update the policy in the direction that increases the expected return $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}$.
However, considering the raw version of expected return, may lead to high variance, therefore **advantage** $A_t$ is considered instead.

The **advantage** $A_t$ measures how good the policy was, after taking action $a_t$ at state $s_t$.

$$A_t = Q(s_t, a_t) - V(s_t)$$

If $A_t > 0$, the action was better than average — increase its probability.
If $A_t < 0$, it was worse than average — decrease its probability.

In **PPO** a different advantage is compute:

$$A_t^{\text{GAE}(\lambda)} = \sum_{l=0}^{T-t} (\gamma \lambda)^l \, \delta_{t+l}$$

where:
- $\delta_t$ is the TD error at state $t$, i.e. the difference between the prediction of the critic and what actually happened.
- $\gamma \in [0,1]$ is the discount factor, it is a hyperparameter that weights the future effects.
- $\lambda$ it is the *GAE parameter*. It is a hyperparameter that balances the computation of advantages.
  When it is set to 0, the advantage is computed considering the next state only. On the other hand, if $\lambda$ is 1, the advantage is estimated considering all the subsequent states.
  Having a value in between results in a blend of these two opposites.

#### TD error ($\delta$)

TD error can be estimated with bootstrapping: 

$$\delta_t = r_t + \gamma \cdot V(s_{t+1}) \cdot (1 - d_t) - V(s_t)$$

where $d_t \in \{0, 1\}$ is a terminal mask ($d_t = 1$ if the episode ended at step $t$, so no bootrapping needed). 

```python
mask = 1 - self.buffer.get('is_terminal', t)
deltas[t] = reward[t] + gamma * mask * V(s_{t+1}) - V(s_t)
```

#### Efficient computation: reverse accumulation

Instead of inefficiently compute the advantages as a mere sum of all advantages, a reverse accumulation has been implemented.
Starting from last time step $T$, the advantage is computed for the previous time steps. This computation is performed for all states until the beginning of the episode:

$$A_t = \delta_t + (\gamma \lambda)(1 - d_t) \cdot A_{t+1}$$

```python
advantage = 0
for t in reversed(range(T)):
    mask = 1 - is_terminal[t]
    advantage = deltas[t] + gamma * lambda * mask * advantage
    advantages[t] = advantage
```

The terminal mask ensures that when an episode ends at step $t$, the advantage does not
go beyond the episode boundaries.

#### Advantage normalisation

After computing all advantages over the rollout, they are normalised to zero mean and unit variance:

$$\hat{A}_t = \frac{A_t - \mu_A}{\sigma_A + \varepsilon}$$

where $\varepsilon = 10^{-8}$ prevents division by zero. This is not part of the original GAE
formulation — it is a practical stabilisation trick. Without it, advantages with large magnitude
produce large gradients that can destabilise training.

#### Returns for the critic

The critic is trained to predict $V(s_t)$. Its target (the label it regresses towards) is
the discounted return from that state:

$$R_t = \sum_{k=0}^{T-t} \gamma^k r_{t+k}$$

computed by another reverse accumulation:

```python
temp = 0
for t in reversed(range(T)):
    temp = reward[t] + gamma * temp
    value_returns[t] = temp
```

Note that the actor uses $R_t^{\text{actor}} = \hat{A}_t + V(s_t)$ as its return target,
while the critic uses the pure discounted return $R_t$. The two targets are related but
computed differently.

### 3.3 PPO clipped objective

#### The core problem PPO solves

In standard policy gradient, nothing stops an update from changing the policy too drastically.
If the calculation of the actor gradients has no boundary, this could lead to divergent trainings. 

PPO fixes this by introducing the clipping on the ratio $r_t$, which quantifies how the policy has changed between two update steps.
If $r_t$ gets too far from 1, its value is clipped to a safer value.

#### The clipped surrogate objective

The PPO objective becomes:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t \left[
\min\left(
r_t \hat{A}_t,\;
\text{clip}(r_t,\, 1-\varepsilon,\, 1+\varepsilon)\, \hat{A}_t
\right)
\right]$$

where $\varepsilon = 0.2$ in this implementation.

There are four cases:

| $\hat{A}_t$ | $r_t$ | What happens |
|---|---|---|
| $> 0$ (good action) | $r_t \leq 1 + \varepsilon$ | Unclipped — update proceeds normally |
| $> 0$ (good action) | $r_t > 1 + \varepsilon$ | **Clipped** — we already pushed this action enough |
| $< 0$ (bad action) | $r_t \geq 1 - \varepsilon$ | Unclipped — update proceeds normally |
| $< 0$ (bad action) | $r_t < 1 - \varepsilon$ | **Clipped** — we already pushed away from this action enough |

It is a non-symmetrical boundary: the ratio is clipped only when the policy has largely improved on the action taken; if, instead, the advantage is highly negative (the taken action should not preferred), the policy is let be free to keep considering that action as bad. 

```python
clip_loss = -np.minimum(
    ratios * advantages,
    np.clip(ratios, 1 - self.epsilon, 1 + self.epsilon) * advantages
)
```

The minus sign is because we are doing gradient **descent** on the loss, but we want to
**maximise** the objective — so we negate it.

#### Entropy bonus

To avoid local optimal policy in early steps, an entropy bonus is added to the loss:

$$\mathcal{H}(\pi) = -\sum_a \pi(a \mid s) \log \pi(a \mid s)$$

The policy has been chosen to be stochastic, however during training its becomes more deterministic. 
Entropy is subtracted to penalize the policy for becoming too deterministic too quickly.

$$\mathcal{L}^{\text{actor}} = \mathcal{L}^{\text{CLIP}} - c_H \cdot \mathcal{H}(\pi)$$

with $c_H = 0.01$.

#### Value loss

In this implementation the actor and critic are separate networks with
separate optimizers.
The critic minimises a standard MSE loss between its prediction and the return target:

$$\mathcal{L}^{\text{critic}} = \mathbb{E}_t \left[ \left( V(s_t) - R_t \right)^2 \right]$$

The total loss is a weighted sum of all three terms:

$$\mathcal{L}^{\text{total}} = \mathcal{L}^{\text{CLIP}} + c_V \cdot \mathcal{L}^{\text{critic}} - c_H \cdot \mathcal{H}(\pi)$$

with $c_V = 0.5$. 



### 3.4 Backpropagation — Actor

This section focuses on the math behind the update of actor and critic networks.

Considering the first one, the derivative of total loss on logits, only considers PPO clip loss and entropy terms, since the value loss does not depend on logits but on returns and state values only.

Here below the steps to derive the formulas for each term separately.

#### Gradient of the clipped loss with respect to logits

The clipped loss for a single timestep is:

$$\mathcal{L}^{\text{CLIP}}_t = -\min\left( r_t \hat{A}_t,\; \text{clip}(r_t, 1-\varepsilon, 1+\varepsilon)\, \hat{A}_t \right)$$

When the sample is **not clipped**, the active term is $-r_t \hat{A}_t$. Therefore:

$$\frac{\partial}{\partial z} (-r_t \hat{A}_t)
= -\frac{\hat{A}_t}{\pi_{\theta_{\text{old}}}(a_t|s_t)} \cdot \frac{\partial \pi_\theta(a_t|s_t)}{\partial z}$$

Now, the derivative of softmax $\partial \pi(a|s) / \partial z$ is needed:
For the chosen action $a_t$ with one-hot encoding $\mathbf{e}_{a_t}$:

$$\frac{\partial \pi(a_t|s)}{\partial z_j} = \pi(a_t|s) \left( \mathbf{e}_{a_t,j} - \pi(j|s) \right)$$

where $\mathbf{e}_{a_t}$ is the one-hot vector for the chosen action $a_t$. It is a vector with zero values everywhere, except for the unitary value at index corresponding to the $a_t$ action.

Substituting back and simplifying:

$$\frac{\partial \mathcal{L}^{\text{CLIP}}_t}{\partial z} = -\hat{A}_t \cdot r_t \cdot (\mathbf{e}_{a_t} - \boldsymbol{\pi})$$

where $\boldsymbol{\pi}$ is the full probability vector over all actions.

Alternatively, when the sample **is clipped**, the gradient is zero — the clip operation has made the
loss flat with respect to $\theta$ in that region, so no update should happen:

$$\frac{\partial \mathcal{L}^{\text{CLIP}}_t}{\partial z} = 0 \quad \text{if clipped}$$

The two clipping conditions are (from the table in 3.3):

```python
clipped1 = (advantages > 0) & (ratios > 1 + epsilon)  # good action, already boosted enough
clipped2 = (advantages < 0) & (ratios < 1 - epsilon)  # bad action, already suppressed enough
dLclip_dz[clipped1] = 0
dLclip_dz[clipped2] = 0
```

#### Gradient of the entropy bonus with respect to logits

The entropy is $\mathcal{H} = -\sum_a \pi(a) \log \pi(a)$. Taking the derivative with respect
to logit $z_j$ and applying the chain rule through the softmax:

$$\frac{\partial \mathcal{H}}{\partial z_j} =
- \pi_j \left( \log \pi_j + 1 \right)
- \sum_a \pi_a \log \pi_a \cdot \pi_j$$

In vector form:

$$\frac{\partial \mathcal{H}}{\partial z} =
- \boldsymbol{\pi} \odot (\log \boldsymbol{\pi} + 1)
- \boldsymbol{\pi} \sum_a \pi_a \log \pi_a$$

```python
dH_dz = (-action_probs * (log_probs + 1)
         - np.sum(action_probs * (np.log(action_probs + 1) * action_probs), axis=1, keepdims=True))
```

#### Combined gradient

The full gradient going into backprop is:

$$\delta = \frac{\partial \mathcal{L}^{\text{CLIP}}}{\partial z} + c_H \cdot \frac{\partial \mathcal{H}}{\partial z}$$

```python
delta = dLclip_dz + entropy_coefficient * dH_dz
```

#### Backprop through the layers

Once $\delta = \partial \mathcal{L} / \partial z$ is computed at the output layer, the rest is
standard backpropagation. For each layer $l$ going backwards:

$$\delta^{(l)} = \delta^{(l+1)} \odot f'(o^{(l)})$$

$$\frac{\partial \mathcal{L}}{\partial W^{(l)}} = (x^{(l)})^\top \delta^{(l)}$$

$$\frac{\partial \mathcal{L}}{\partial b^{(l)}} = \sum_i \delta^{(l)}_i$$

$$\delta^{(l-1)} = \delta^{(l)} (W^{(l)})^\top$$

where $f'(o^{(l)})$ is the activation derivative cached during the forward pass
(for tanh: $1 - \tanh^2(x)$), and $x^{(l)}$ is the layer input also cached during forward.

```python
for layer in reversed(actor_network.w.keys()):
    delta = delta * actor_network.activation_derivative(cache[layer]['output'], layer)
    w_grad[layer] = cache[layer]['input'].T @ delta
    b_grad[layer] = np.sum(delta, axis=0, keepdims=True)
    delta = delta @ actor_network.w[layer].T
```

### 3.5 Backpropagation — Critic

The critic minimises an MSE loss:

$$\mathcal{L}^{\text{critic}} = \frac{1}{N} \sum_t (V(s_t) - R_t)^2$$

The derivative with respect to the critic output $v = V(s_t)$ is:

$$\frac{\partial \mathcal{L}^{\text{critic}}}{\partial v} = \frac{2}{N} (V(s_t) - R_t)$$

```python
delta = 2 * (state_values - returns) / len(indices)
```

From here the backprop through the layers is identical to the actor — same loop,
same formulas for $W$ and $b$ gradients. The only difference is the sign of the
parameter update: the critic does **gradient descent** (minimises the loss),
while the actor does **gradient ascent** (maximises the objective).

In the code this is handled by a sign flip inside `update_parameters`:

```python
sign = +1 if self.name == 'actor' else -1
self.w[layer] += sign * optimizer.optimize_gradient(w_grad[layer], ...)
```

### 3.6 Adam optimizer

Once the gradients are computed, they need to be used to update the weights.
The simplest option would be plain gradient descent:

$$\theta \leftarrow \theta - \alpha \cdot g_t$$

#### The two moment estimates

Adam maintains two running averages for each parameter:

**First moment** $m_t$ — an exponential moving average of the gradients.
It smooths out noise and gives a sense of the direction the gradient has been pointing on average:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$

**Second moment** $v_t$ — an exponential moving average of the squared gradients.
It tracks how much the gradient has been varying — large $v_t$ means the gradient
has been noisy or large, small $v_t$ means it has been stable and small:

$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$

With $\beta_1 = 0.9$ and $\beta_2 = 0.999$, $m_t$ adapts quickly (forgets old gradients
in ~10 steps) while $v_t$ adapts slowly (remembers the last ~1000 steps).

#### Bias correction

Both $m_t$ and $v_t$ are initialised at zero. At the start of training this makes them
biased towards zero — $m_1 = (1 - 0.9) \cdot g_1 = 0.1 \cdot g_1$ is much smaller than
$g_1$ itself. The fix is to divide by the bias correction factor:

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

As $t$ grows, $\beta^t \to 0$ and the correction factor $\to 1$, so it only matters
in the first few steps.

Note that in my implementation the correction uses a fixed $\beta$ rather than $\beta^t$,
which means the bias correction does not decay over time.

#### The update rule

The final parameter update divides the smoothed gradient by the square root of the
smoothed squared gradient:

$$\theta_t = \theta_{t-1} - \alpha \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}$$

```python
self.m[layer][p] = beta1 * self.m[layer][p] + (1 - beta1) * gradient
self.v[layer][p] = beta2 * self.v[layer][p] + (1 - beta2) * gradient**2
m_hat = self.m[layer][p] / (1 - beta1)
v_hat = self.v[layer][p] / (1 - beta2)
update = -learning_rate * m_hat / (np.sqrt(v_hat) + 1e-8)
```

The $\varepsilon = 10^{-8}$ in the denominator prevents division by zero when a gradient
has been exactly zero for many steps.

## 4. Algorithm

With all the pieces from section 3 in place, here is how they fit together.
The full training loop follows the standard PPO structure: collect experience with the
current policy, freeze it as the "old policy", then update the current policy against it
for several epochs.

---

```
Initialise actor network π_θ and critic network V_φ

for each episode:

    old_policy ← deepcopy(agent)          # freeze the current policy

    ── ROLLOUT PHASE ──────────────────────────────────────────────
    for t = 1 ... rollout_steps:
        z         ← actor.forward(s_t)         # raw logits
        π(·|s_t)  ← softmax(z)                 # action probabilities
        a_t       ~ π(·|s_t)                   # sample action
        V(s_t)    ← critic.forward(s_t)        # state value

        s_{t+1}, r_t, done ← env.step(a_t)

        store (s_t, a_t, r_t, V(s_t), log π(a_t|s_t)) in buffer

        if done: reset environment

    ── ADVANTAGE & RETURNS ────────────────────────────────────────
    for t = T ... 1:                           # reverse accumulation
        δ_t ← r_t + γ · V(s_{t+1}) · (1-d_t) - V(s_t)
        A_t ← δ_t + γλ · (1-d_t) · A_{t+1}

    A ← normalise(A)                           # zero mean, unit variance
    R ← discounted returns                     # critic targets

    ── UPDATE PHASE ───────────────────────────────────────────────
    for epoch = 1 ... update_epochs:

        indices ← random minibatch from buffer

        recompute π_θ(a|s), V_φ(s) on minibatch   # with current (updated) networks

        r_t ← π_θ(a_t|s_t) / π_old(a_t|s_t)      # probability ratio

        L_CLIP ← -mean[ min(r_t · A_t,
                            clip(r_t, 1-ε, 1+ε) · A_t) ]  # PPO loss

        L_H    ← -mean[ Σ_a π(a|s) log π(a|s) ]            # entropy bonus
        L_V    ←  mean[ (V_φ(s) - R)² ]                    # value loss

        ∂L/∂z_actor  ← backprop through L_CLIP + c_H · L_H  (section 3.4)
        ∂L/∂z_critic ← backprop through L_V                 (section 3.5)

        θ ← θ + Adam(∂L/∂z_actor)     # gradient ascent on actor
        φ ← φ - Adam(∂L/∂z_critic)    # gradient descent on critic
```

---

A few things worth noting:

The **rollout is collected entirely with `old_policy`**, not with the network being updated.
This is the key PPO idea — the data is collected once, then reused for multiple gradient
steps. The ratio $r_t$ keeps track of how much the current policy has drifted from the one
that collected the data, and the clip prevents that drift from getting too large.

The **advantage is computed after the rollout**, not during. This is necessary because
the GAE formula requires the full sequence of TD errors before it can start the
reverse accumulation.

The **minibatch is resampled at each epoch**. This means different subsets of the rollout
buffer are used in each update step, which adds a small amount of stochasticity and
prevents overfitting to any particular transition.

The **actor and critic are updated separately** with their own gradient flows and their
own Adam instances.

## 5. Experiments

### 5.1 Experimental setup

To understand how each hyperparameter affects training, I ran a **one-parameter-at-a-time** sweep:
one parameter is varied while all others are kept fixed at the baseline values below.

| Parameter | Baseline value |
|---|---|
| Learning rate | `3e-4` |
| Discount factor γ | `0.99` |
| Clip coefficient ε | `0.2` |
| Entropy coefficient $c_H$ | `0.01` |
| Lambda λ | `0.95` |
| Value loss coefficient $c_V$ | `0.5` |
| Batch size | `64` |
| Rollout steps | `2048` |
| Update epochs | `4` |

Each configuration was run over **5 random seeds** (0, 1, 2, 4, 5) for 2000 episodes.
The curves shown in the plots are the **average across seeds**, with the shaded area
representing ±1 standard deviation. Using multiple seeds is important because a single
run can be misleading — a lucky initialisation might make a bad hyperparameter look good,
or a unlucky one might unfairly penalise a good configuration.

The metrics tracked for each run are:

| Metric | What it tells us |
|---|---|
| **Reward** | Overall learning progress — the main performance indicator |
| **Policy Loss** | Whether the actor objective is being optimised correctly |
| **Value Loss** | How well the critic is learning to estimate $V(s)$ |
| **Entropy** | Whether the policy is exploring or collapsing prematurely |
| **Clip Fraction** | How often the PPO clip is actually activating |
| **Ratio** | How much the policy is drifting from the old policy |
| **Advantage** | Stability of the GAE estimates |

### 5.2 Learning rate

Values: `1e-4`, `3e-4` (baseline), `1e-3`.

<div style='text-align: center;'>
<img src='results/learning_rate_Reward.png' width='500'/>
<img src='results/learning_rate_Value Loss.png' width='500'/>
<img src='results/learning_rate_Entropy.png' width='500'/>
<img src='results/learning_rate_Clip Fraction.png' width='500'/>
</div>

The learning rate is the parameter with the most visible impact across all metrics.

**Reward** — the three values produce clearly different trajectories. `1e-3` converges
fastest and reaches the highest reward early on (~250 by episode 200), but becomes
increasingly unstable in the second half of training, with large oscillations and high
variance across seeds. `3e-4` is slower to start but steadier — it keeps improving
gradually and with less variance. `1e-4` is simply too slow: after 1000 episodes it
is still around 125, well below the other two.

**Value Loss** — All three curves stay close to 0.5
throughout training. Looking more carefully though, lower learning rates produce noticeably
smoother curves with fewer spikes — `1e-4` is the flattest, `1e-3` has the most
pronounced oscillations. Higher learning rate means more aggressive updates, which leads to less stable estimates.

**Entropy** — all three curves decrease monotonically, meaning the policy is becoming
more confident over time in all cases. What differs is the rate: `1e-3` reaches its
minimum entropy much faster, confirming that a higher learning rate pushes the policy
towards determinism more quickly. Whether this is desirable depends on whether the
policy has had enough time to explore first.

**Clip Fraction** — essentially zero across all learning rates. As further analysis confirms, CartPole-v1 environment has not significant differents in rewards,
which leads to small policy ratios, which finally leads in having the PPO clipping to never activate.

**Conclusion**: `3e-4`seems to be the right choice for this environment — fast enough to learn
within a reasonable number of episodes, stable enough to keep improving without
collapsing. `1e-3` is an option if fast convergence matters more than stability.

### 5.3 Clip coefficient (ε)

Values: `0.1`, `0.2` (baseline), `0.3`.

<div style='text-align: center;'>
<img src='results/clip_coefficient_Reward.png' width='500'/>
<img src='results/clip_coefficient_Clip Fraction.png' width='500'/>
<img src='results/clip_coefficient_Ratio.png' width='500'/>
</div>

The three curves are perfectly overlapping across every metric. 
Changing ε from 0.1 to 0.3 has zero measurable effect.

This is not a bug — it is a direct consequence of what we observed during debugging.
The ratio $r_t = \pi_\theta / \pi_{\theta_{\text{old}}}$ never moves far from 1.0 on
CartPole: the policy updates are too small for the ratio to exit the interval
$[1-\varepsilon, 1+\varepsilon]$, regardless of how wide or narrow that interval is.
The clip fraction confirms this — it is exactly zero throughout training for all three values.


**Conclusion**: on CartPole, ε is irrelevant. The environment is too simple for the PPO
clipping mechanism to ever be needed. To see ε actually matter, a more complex
environment is required — one where policy updates are large enough to push the ratio
outside the clipping range.

### 5.5 Lambda (λ)

Values: `0.9`, `0.95` (baseline), `0.99`.

<div style='text-align: center;'>
<img src='results/lambda_Reward.png' width='500'/>
<img src='results/lambda_Advantage.png' width='500'/>
<img src='results/lambda_Value Loss.png' width='500'/>
</div>


**Reward** — the three curves separate early and stay separated throughout training.
`λ = 0.99` reaches the highest reward (~260 by episode 1000) and keeps improving,
`λ = 0.95` settles around 210, and `λ = 0.9` plateaus at 150 — noticeably lower
than the other two. The variance across seeds is similar.

A higher λ gives more weight to rewards further in the future, producing a less 
biased but higher variance estimate of the advantage. 
A low λ like 0.9 discounts future rewards too aggressively, making the advantage 
estimate shortsighted and slowing down learning.

**Advantage** — the three curves oscillate around zero with similar magnitude, which
is expected after normalisation. No clear trend can be noticed.

**Value Loss** — As the number of played episodes increases `λ = 0.99`, the value loss
behaviour seems to dampen its oscillations. Whereas, lower `λ` exhibit more instable behavior

**Conclusion**: Higher values work better here because CartPole rewards persistence — the longer the pole stays up,
the higher the return. A policy that can see further into the future learns this
faster. `λ = 0.99` outperforms the baseline `0.95`, suggesting the baseline could
be tuned further.

### 5.6 Value loss coefficient ($c_V$)

Values: `0.1`, `0.5` (baseline), `1.0`.

<div style='text-align: center;'>
<img src='results/value_loss_coefficient_Reward.png' width='500'/>
<img src='results/value_loss_coefficient_Value Loss.png' width='500'/>
<img src='results/value_loss_coefficient_Advantage.png' width='500'/>
</div>

**Reward** — the three curves are again perfectly overlapping, similar to what we
saw with the clip coefficient. Changing $c_V$ by an order of magnitude has no
measurable effect on the reward trajectory.

**Value Loss** — this is where the effect is visible.
The reported value loss is $c_V \cdot \text{MSE}$, so the three curves are naturally
scaled by their respective coefficients: `0.1` sits around 0.1, `0.5` around 0.5,
and `1.0` around 1.0. The underlying MSE is identical across all three — only the
scale changes. This confirms that $c_V$ is acting purely as a scaling factor on the
loss value, without affecting the actual quality of the critic's predictions.

**Advantage** — fully overlapping, as expected. The advantage depends on how well
the critic estimates $V(s)$, and since all three critics learn equally well,
the advantage estimates are indistinguishable.

**Conclusion**: in this implementation $c_V$ does not matter because the actor and
critic are updated independently with separate optimizers. The coefficient would
only be meaningful in a shared-backbone architecture, where both losses are summed
before a single backward pass.

### 5.7 Rollout steps

Values tested: `512`, `2048` (baseline), `4096`.

<div style='text-align: center;'>
<img src='results/rollout_steps_Reward.png' width='500'/>
<img src='results/rollout_steps_Advantage.png' width='500'/>
<img src='results/rollout_steps_Policy Loss.png' width='500'/>
</div>

**Reward** — longer rollouts produce better results. `4096` reaches the highest
reward and keeps a slight edge over `2048` throughout training. `512` plateaus
earlier and lower, around 160, while the other two continue improving past 200.

The intuition is straightforward: a longer rollout gives the GAE computation more
transitions to work with, producing more reliable advantage estimates. With only
512 steps, many rollouts end before the buffer is full (CartPole episodes can
terminate early), so the advantage estimates are noisier and the update signal
is weaker.

**Advantage** — all three curves oscillate around zero with similar amplitude.
Interestingly, `512` shows slightly higher variance in the advantage estimates,
consistent with the idea that shorter rollouts produce noisier GAE values.

**Policy Loss** — `512` has noticeably higher oscillations than the other two,
which is another sign that shorter rollouts make the update signal less stable.
With fewer transitions per update, the minibatch is a less representative sample
of the current policy's behaviour.

**Conclusion**: more rollout steps help, but with diminishing returns — `4096` is
only marginally better than `2048`. The main cost of longer rollouts is wall-clock
time: each episode takes longer to collect data before any update happens.
`2048` is a reasonable default that balances sample quality with training speed.

### 5.8 Update epochs

Values tested: `3`, `4` (baseline), `5`.

<div style='text-align: center;'>
<img src='results/update_epochs_Reward.png' width='500'/>
<img src='results/update_epochs_Policy Loss.png' width='500'/>
<img src='results/update_epochs_Clip Fraction.png' width='500'/>
</div>

**Reward** — more update epochs produce a small but consistent improvement.
`5` and `4` are close to each other and both outperform `3`, which plateaus
slightly lower and earlier. 

**Policy Loss** — `3` epochs shows higher oscillations and a noticeable spike
late in training, while `4` and `5` are more stable. Fewer updates per rollout
means the policy gradient signal is noisier, since each minibatch is a smaller
fraction of the available experience.

**Clip Fraction** — zero across all values, consistent with every other experiment.

**Conclusion**: within this small range, more epochs is better. The range tested
here is narrow (3–5) — in practice PPO is often run with 10 or more epochs on
harder environments, where the clip becomes essential to prevent overfitting
to the current batch of experience.

## 6. Conclusions

### 6.1 What I learned

Implementing PPO from scratch was a different experience from reading about it.
The algorithm looks clean on paper, but working through every component by hand
surfaced a number of things that a library would have hidden.

The most instructive moment was deriving the gradient of the clipped loss by hand.
Understanding that the clip zeroes out the gradient — rather than just clamping the
loss value — made the mechanism much clearer. The same goes for the sign asymmetry
in the actor update (gradient ascent) vs the critic (gradient descent): a one-line
difference in code that is easy to get wrong and hard to debug once it is.

The GAE reverse accumulation was another case where the implementation forced a
deeper understanding. Knowing the formula is one thing; writing the loop and making
sure the terminal mask is in the right place is another.

### 6.2 What the experiments showed

The hyperparameter sweep produced a clear ranking of which parameters matter on CartPole:

| Parameter | Impact | Notes |
|---|---|---|
| **λ** | High | Most impactful — higher values consistently better |
| **Learning rate** | High | Strong effect on speed and stability |
| **Rollout steps** | Medium | More steps help, with diminishing returns |
| **Update epochs** | Medium | More epochs help within the range tested |
| **Entropy coefficient** | Low | Affects entropy but not reward on this environment |
| **Clip coefficient** | None | PPO clip never activates on CartPole |
| **Value loss coefficient** | None | Irrelevant with separate networks |

The most surprising result was the clip coefficient: despite being the defining
feature of PPO, it had zero effect here. The policy updates on CartPole are simply
too small for the ratio $r_t$ to ever leave the clipping range. This is not a bug —
it is a property of the environment. A harder environment would tell a different story.

### 6.3 Limitations and next steps

A few things are worth improving if this project were to go further:

- **Test on harder environments** — LunarLander or a MuJoCo task would stress-test
  the PPO clipping and entropy bonus in ways CartPole cannot.
- **Shared backbone** — implementing the actor and critic with shared layers would
  make $c_V$ meaningful and is a more common architecture in practice.
- **Learning rate schedule** — a decaying learning rate could combine the fast
  early convergence of `1e-3` with the late-stage stability of `3e-4`.
- **Broader λ sweep** — since `λ = 0.99` outperformed the baseline, it would be
  worth testing values closer to 1 to find the actual optimum for this environment.